In [ ]:
from src.config import CONFIG
from src.loaders import load_tradingview_csv, assign_sessions
from src.engine import EngineState, update_engine_state
from src.replay import run_replay

### Section 12 — Replay Runner

Replay mode steps through historical data one bar at a time, simulating exactly the information state an operator would have seen in real time.

#### Look-ahead bias controls
- The engine state at bar *t* contains only information from bars 0 to *t*
- Outcome labels are not shown until bar *t + N* is reached
- The probability table must be calibrated on separate historical data

The replay function is a Python generator — it yields one `EngineState` per bar. This makes it easy to connect to a notebook display, logging loop, or future chart UI.

In [ ]:
print("✅ Replay notebook setup loaded")
print(f"   Instrument : {CONFIG['instrument']}")
print(f"   Timeframe  : {CONFIG['timeframe']}")
print("   run_replay() available")

In [ ]:
# ── Demo: step through 10 bars of replay and print state ──
print("Replay demo (first 10 bars):")
print(f"{'Bar':>4} {'Zone':>5} {'Z-Score':>8} {'P(MR)':>7} {'P(CONT)':>8} {'Trend':>8}")
print("-" * 50)

for i, state in enumerate(
    run_replay(
        df,
        CONFIG,
        prob_table,
        EngineState=EngineState,
        update_engine_state=update_engine_state,
        speed=1e9,
    )
):
    if i >= 10:
        break
    mr = state.probabilities.get('MR', {}).get('prob', 0)
    cont = state.probabilities.get('CONT', {}).get('prob', 0)
    trend = state.context.get('trend_bin', '?')
    print(f"{i:>4} {state.zone:>5} {state.z_score:>8.3f} {mr:>7.1%} {cont:>8.1%} {trend:>8}")